# 🔍 Named Entity Recognition System — Fully Self-Contained Notebook

This single notebook contains **everything** needed to run end-to-end —
data loading, models, training, evaluation, comparison, visualization, and
the Gradio app — with **no dependency on any other project file**. You can
upload just this `.ipynb` to **Kaggle**, **Google Colab**, or any Jupyter
environment and run every cell top to bottom.

Pipeline:
1. Setup
2. Load & explore CoNLL-2003 (IOB2 tagging)
3. OOV handling: character-level vocab
4. Pretrained GloVe embeddings (optional)
5. Preprocessing / batching
6. Model definitions: Char-CNN, from-scratch CRF, LSTM / BiLSTM / BiLSTM+CRF
7. Training loop + seqeval evaluation
8. Train LSTM, BiLSTM, BiLSTM+CRF
9. Fine-tune a Transformer (HuggingFace BERT/DistilBERT)
10. Compare all four architectures
11. Visualize predicted entity spans (color-coded, inline)
12. Launch the Gradio deployment app

> 💡 Best run on **Google Colab** or **Kaggle** with a **GPU runtime** and
> internet access enabled, since step 2 and step 9 download data / weights
> from the HuggingFace Hub.


## 1. Setup

In [27]:
# Uncomment on a fresh environment (Kaggle usually already has these):
# !pip install -q torch transformers datasets seqeval gradio pandas tqdm

import os, re, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from IPython.display import display, HTML

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Device:", device)


Torch: 2.10.0+cu128 | Device: cuda


## 2. Load & explore the dataset (CoNLL-2003, IOB2 tagging)

CoNLL-2003 ships pre-tagged with the IOB2 scheme (`O`, `B-PER`/`I-PER`,
`B-ORG`/`I-ORG`, `B-LOC`/`I-LOC`, `B-MISC`/`I-MISC`). The canonical
`conll2003` HF dataset script has become unreliable (needs
`trust_remote_code`, sometimes gated), so we try a few reliable sources in
order and fall back to a small synthetic dataset if none are reachable —
so the rest of the notebook always runs as a structural smoke test even
fully offline.


In [28]:
LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG",
              "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}
ENTITY_TYPES = ["PER", "ORG", "LOC", "MISC"]


def make_synthetic_dataset(n_train=300, n_val=60, n_test=60, seed=0):
    rnd = random.Random(seed)
    persons = ["John Smith", "Angela Merkel", "Elon Musk", "Marie Curie", "Barack Obama"]
    orgs = ["Google", "United Nations", "Ministry of Defence", "FC Barcelona", "NASA"]
    locs = ["Paris", "New York", "Berlin", "Mount Everest", "Nile River"]
    miscs = ["Olympics", "World Cup", "Nobel Prize", "Renaissance", "Christianity"]
    fillers = ["the", "a", "in", "on", "met", "with", "announced", "yesterday", "said",
               "reported", "visited", "won", "held", "at", "during", "for", "about"]

    def entity_tokens(text, etype):
        words = text.split()
        return words, [f"B-{etype}"] + [f"I-{etype}"] * (len(words) - 1)

    def gen_sentence():
        tokens, tags = [], []
        pools = [("PER", persons), ("ORG", orgs), ("LOC", locs), ("MISC", miscs)]
        for _ in range(rnd.randint(3, 6)):
            tokens.append(rnd.choice(fillers)); tags.append("O")
        for _ in range(rnd.randint(1, 3)):
            etype, pool = rnd.choice(pools)
            words, etags = entity_tokens(rnd.choice(pool), etype)
            tokens.extend(words); tags.extend(etags)
            for _ in range(rnd.randint(1, 4)):
                tokens.append(rnd.choice(fillers)); tags.append("O")
        return {"tokens": tokens, "ner_tags": [LABEL2ID[t] for t in tags]}

    return {"train": [gen_sentence() for _ in range(n_train)],
            "validation": [gen_sentence() for _ in range(n_val)],
            "test": [gen_sentence() for _ in range(n_test)]}


def try_load_real_conll2003():
    from datasets import load_dataset

    last_err = None
    # Try a couple of known-good sources in order.
    for repo, kwargs in [
        ("lhoestq/conll2003", {}),
        ("conll2003", {"trust_remote_code": True}),
        ("eriktks/conll2003", {"trust_remote_code": True}),
    ]:
        try:
            print(f"Trying dataset repo: {repo} ...")
            ds = load_dataset(repo, **kwargs)
            feat = ds["train"].features["ner_tags"]
            hf_names = feat.feature.names if hasattr(feat, "feature") else None

            def remap(example):
                if hf_names is not None:
                    example["ner_tags"] = [LABEL2ID[hf_names[t]] for t in example["ner_tags"]]
                return example

            if hf_names is not None:
                ds = ds.map(remap)

            raw = {split: list(ds[split]) for split in ["train", "validation", "test"]}
            print(f"Loaded REAL CoNLL-2003 from '{repo}'.")
            return raw
        except Exception as e:
            print(f"  failed ({type(e).__name__}: {str(e)[:120]})")
            last_err = e
            continue
    raise last_err


try:
    raw = try_load_real_conll2003()
except Exception as e:
    print(f"\nCould not load any real CoNLL-2003 source ({e}). Falling back to SYNTHETIC data.")
    raw = make_synthetic_dataset()


def dataset_stats(split_data):
    vocab, tag_counts = Counter(), Counter()
    for ex in split_data:
        for tok in ex["tokens"]:
            vocab[tok.lower()] += 1
        for tid in ex["ner_tags"]:
            tag_counts[ID2LABEL[tid]] += 1
    return {"num_sentences": len(split_data), "vocab_size": len(vocab),
            "tag_distribution": dict(tag_counts)}

for split in ["train", "validation", "test"]:
    print(split, dataset_stats(raw[split]))


Trying dataset repo: lhoestq/conll2003 ...


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'eriktks/conll2003' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  failed (RuntimeError: Dataset scripts are no longer supported, but found conll2003.py)
Trying dataset repo: eriktks/conll2003 ...


README.md: 0.00B [00:00, ?B/s]

conll2003.py: 0.00B [00:00, ?B/s]

  failed (RuntimeError: Dataset scripts are no longer supported, but found conll2003.py)

Could not load any real CoNLL-2003 source (Dataset scripts are no longer supported, but found conll2003.py). Falling back to SYNTHETIC data.
train {'num_sentences': 300, 'vocab_size': 51, 'tag_distribution': {'O': 2805, 'B-MISC': 146, 'I-MISC': 58, 'B-LOC': 144, 'I-LOC': 84, 'B-PER': 154, 'I-PER': 154, 'B-ORG': 153, 'I-ORG': 113}}
validation {'num_sentences': 60, 'vocab_size': 51, 'tag_distribution': {'O': 579, 'B-ORG': 32, 'I-ORG': 25, 'B-PER': 30, 'I-PER': 30, 'B-MISC': 23, 'I-MISC': 9, 'B-LOC': 38, 'I-LOC': 26}}
test {'num_sentences': 60, 'vocab_size': 51, 'tag_distribution': {'O': 600, 'B-PER': 28, 'I-PER': 28, 'B-LOC': 34, 'I-LOC': 21, 'B-MISC': 32, 'I-MISC': 15, 'B-ORG': 32, 'I-ORG': 25}}


In [29]:
from datasets import load_dataset

# تحميل النسخة المحدثة والآمنة مباشرة التي لا تعتمد على سكريبت conll2003.py
dataset = load_dataset("lhoestq/conll2003")

# استعراض البيانات للتأكد من نجاح التحميل
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [30]:
import pandas as pd

# تحويل الأقسام الثلاثة إلى جداول
df_train = pd.DataFrame(dataset['train'])
df_val = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])

# لعرض جدول التدريب كمثال
print("Data Size:", df_train.shape)
df_train.head(10) # يعرض أول 10 أسطر كعينة سريعة


Data Size: (14041, 5)


,id,tokens,pos_tags,chunk_tags,ner_tags
0,0,"[EU, rejects, German, call, to, boycott, Briti...","[22, 42, 16, 21, 35, 37, 16, 21, 7]","[11, 21, 11, 12, 21, 22, 11, 12, 0]","[3, 0, 7, 0, 0, 0, 7, 0, 0]"
1,1,"[Peter, Blackburn]","[22, 22]","[11, 12]","[1, 2]"
2,2,"[BRUSSELS, 1996-08-22]","[22, 11]","[11, 12]","[5, 0]"
3,3,"[The, European, Commission, said, on, Thursday...","[12, 22, 22, 38, 15, 22, 28, 38, 15, 16, 21, 3...","[11, 12, 12, 21, 13, 11, 11, 21, 13, 11, 12, 1...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ..."
4,4,"[Germany, 's, representative, to, the, Europea...","[22, 27, 21, 35, 12, 22, 22, 27, 16, 21, 22, 2...","[11, 11, 12, 13, 11, 12, 12, 11, 12, 12, 12, 1...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ..."
5,5,"["", We, do, n't, support, any, such, recommend...","[0, 28, 41, 30, 37, 12, 16, 21, 15, 28, 41, 30...","[0, 11, 21, 22, 22, 11, 12, 12, 17, 11, 21, 22...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6,6,"[He, said, further, scientific, study, was, re...","[28, 38, 16, 16, 21, 38, 40, 10, 15, 28, 38, 4...","[11, 21, 11, 12, 12, 21, 22, 0, 17, 11, 21, 22...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7,7,"[He, said, a, proposal, last, month, by, EU, F...","[28, 38, 12, 21, 16, 21, 15, 22, 22, 22, 22, 2...","[11, 21, 11, 12, 11, 12, 13, 11, 12, 12, 12, 1...","[0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 1, 2, 0, 0, 0, ..."
8,8,"[Fischler, proposed, EU-wide, measures, after,...","[17, 40, 22, 42, 15, 24, 15, 22, 10, 22, 43, 1...","[11, 12, 12, 21, 13, 11, 13, 11, 12, 12, 11, 1...","[1, 0, 7, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0, 0, ..."
9,9,"[But, Fischler, agreed, to, review, his, propo...","[10, 22, 38, 35, 37, 29, 21, 15, 12, 22, 27, 2...","[0, 11, 21, 22, 22, 11, 12, 13, 11, 12, 11, 12...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, ..."


In [31]:
# Peek at a few tagged sentences
for ex in raw["train"][:3]:
    print(" ".join(f"{t}/{ID2LABEL[l]}" for t, l in zip(ex["tokens"], ex["ner_tags"])))


at/O a/O said/O about/O for/O held/O Nobel/B-MISC Prize/I-MISC about/O met/O New/B-LOC York/I-LOC said/O
reported/O on/O in/O visited/O Elon/B-PER Musk/I-PER visited/O announced/O for/O during/O Paris/B-LOC in/O
the/O for/O visited/O yesterday/O visited/O in/O United/B-ORG Nations/I-ORG during/O in/O


In [32]:
from collections import defaultdict
from datasets import DatasetDict

langs = ["de", "fr", "it", "en"]
fracs = [0.629, 0.229, 0.084, 0.059]

panx_ch = defaultdict(DatasetDict)
for lang, frac in zip(langs, fracs):
  ds = load_dataset('xtreme', f'PAN-X.{lang}')
  for split in ds:
    panx_ch[lang][split] = (
        ds[split].shuffle(seed=0).select(range(int(frac*ds[split].num_rows))))

README.md: 0.00B [00:00, ?B/s]

PAN-X.de/train-00000-of-00001.parquet:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

PAN-X.de/validation-00000-of-00001.parqu(…):   0%|          | 0.00/590k [00:00<?, ?B/s]

PAN-X.de/test-00000-of-00001.parquet:   0%|          | 0.00/588k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

PAN-X.en/validation-00000-of-00001.parqu(…):   0%|          | 0.00/472k [00:00<?, ?B/s]

PAN-X.en/test-00000-of-00001.parquet:   0%|          | 0.00/472k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [33]:
import pandas as pd
pd.DataFrame({lang: [panx_ch[lang]["train"].num_rows] for lang in langs},
 index=["Number of training examples"])

,de,fr,it,en
Number of training examples,12580,4580,1680,1180


In [34]:
panx_ch[lang]["train"]['ner_tags'][0]

[0, 0, 3, 4, 0, 0]

In [35]:
element = panx_ch["de"]["train"][0]
for key, value in element.items():
 print(f"{key}: {value}")

tokens: ['2.000', 'Einwohnern', 'an', 'der', 'Danziger', 'Bucht', 'in', 'der', 'polnischen', 'Woiwodschaft', 'Pommern', '.']
ner_tags: [0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0]
langs: ['de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de']


In [36]:
for key, value in panx_ch["de"]["train"].features.items():
  print(f"{key}: {value}")

tokens: List(Value('string'))
ner_tags: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))
langs: List(Value('string'))


In [37]:
tags = panx_ch["de"]["train"].features['ner_tags'].feature
tags

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])

In [38]:
def create_tag_names(batch):
  return {'ner_tags_str':[tags.int2str(idx) for idx in batch['ner_tags']]}

panx_de = panx_ch['de'].map(create_tag_names)
de_example = panx_de["train"][0]
pd.DataFrame([de_example["tokens"], de_example["ner_tags_str"]],
['Tokens', 'Tags'])

Map:   0%|          | 0/12580 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


In [39]:
from collections import Counter

split2freqs = defaultdict(Counter)

for split, datast in panx_de.items():
  for row in datast['ner_tags_str']:
    for tag in row:
      if tag.startswith('B'):
        tag_type = tag.split('-')[1]
        split2freqs[split][tag_type]+=1
pd.DataFrame.from_dict(split2freqs, orient="index")

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


## 3. OOV handling — character-level vocabulary

A word missing from GloVe still has informative *spelling* features
(capitalization, suffixes, digits, length). We build a character
vocabulary here; the char-CNN model defined in Section 6 turns this into a
learned embedding per word, giving the tagger signal even for unseen words.


In [40]:
PAD, UNK = "<PAD>", "<UNK>"

class WordVocab:
    def __init__(self, sentences, min_freq=1, lowercase=True):
        self.lowercase = lowercase
        counter = Counter()
        for sent in sentences:
            for tok in sent:
                counter[tok.lower() if lowercase else tok] += 1
        self.itos = [PAD, UNK] + [w for w, c in counter.items() if c >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, tokens):
        return [self.stoi.get(t.lower() if self.lowercase else t, self.stoi[UNK]) for t in tokens]

    def __len__(self):
        return len(self.itos)


class CharVocab:
    def __init__(self, sentences):
        chars = set()
        for sent in sentences:
            for tok in sent:
                chars.update(list(tok))
        self.itos = [PAD, UNK] + sorted(chars)
        self.stoi = {c: i for i, c in enumerate(self.itos)}

    def encode_word(self, word, max_len=20):
        ids = [self.stoi.get(c, self.stoi[UNK]) for c in word[:max_len]]
        ids += [self.stoi[PAD]] * (max_len - len(ids))
        return ids

    def encode_sentence(self, tokens, max_len=20):
        return [self.encode_word(t, max_len) for t in tokens]

    def __len__(self):
        return len(self.itos)


train_sentences = [ex["tokens"] for ex in raw["train"]]
word_vocab = WordVocab(train_sentences)
char_vocab = CharVocab(train_sentences)
print("Word vocab size:", len(word_vocab), "| Char vocab size:", len(char_vocab))


Word vocab size: 53 | Char vocab size: 42


## 4. Pretrained word embeddings (GloVe)

Optional but recommended: download GloVe once, then point `GLOVE_PATH` at
the `.txt` file. Words in our vocab not found in GloVe are randomly
initialized (combined with the char-CNN above, this is how OOV is handled).


In [41]:
# Optional: download GloVe vectors (uncomment on Colab/Kaggle with internet)
# !wget -nc https://nlp.stanford.edu/data/glove.6B.zip
# !unzip -n glove.6B.zip -d glove/

GLOVE_PATH = "glove/glove.6B.100d.txt"   # set to None to use random init only
WORD_EMB_DIM = 100

def load_pretrained_embeddings(vocab, embedding_path, embedding_dim=100):
    matrix = np.random.uniform(-0.25, 0.25, (len(vocab), embedding_dim)).astype("float32")
    matrix[vocab.stoi[PAD]] = np.zeros(embedding_dim, dtype="float32")
    if not embedding_path or not os.path.exists(embedding_path):
        print(f"[embeddings] '{embedding_path}' not found -- using random init.")
        return matrix
    found = 0
    with open(embedding_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            if parts[0] in vocab.stoi:
                vec = np.asarray(parts[1:], dtype="float32")
                if vec.shape[0] == embedding_dim:
                    matrix[vocab.stoi[parts[0]]] = vec
                    found += 1
    print(f"[embeddings] matched {found}/{len(vocab)} vocab words to pretrained vectors.")
    return matrix

embedding_matrix = load_pretrained_embeddings(word_vocab, GLOVE_PATH, WORD_EMB_DIM)
print("Embedding matrix shape:", embedding_matrix.shape)


[embeddings] 'glove/glove.6B.100d.txt' not found -- using random init.
Embedding matrix shape: (53, 100)


## 5. Preprocessing / batching for the scratch models

In [42]:
PAD_LABEL_ID = -100

class NERScratchDataset(Dataset):
    def __init__(self, examples, word_vocab, char_vocab, max_word_len=20):
        self.examples, self.word_vocab, self.char_vocab = examples, word_vocab, char_vocab
        self.max_word_len = max_word_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        tokens, tags = ex["tokens"], ex["ner_tags"]
        return {
            "word_ids": torch.tensor(self.word_vocab.encode(tokens), dtype=torch.long),
            "char_ids": torch.tensor(self.char_vocab.encode_sentence(tokens, self.max_word_len), dtype=torch.long),
            "labels": torch.tensor(tags, dtype=torch.long),
            "length": len(tokens),
        }


def collate_fn(batch, pad_word_id=0, pad_char_id=0, pad_label_id=PAD_LABEL_ID):
    max_len = max(item["length"] for item in batch)
    max_word_len = batch[0]["char_ids"].shape[1]
    word_ids, char_ids, labels, masks, lengths = [], [], [], [], []
    for item in batch:
        L = item["length"]; pad_amt = max_len - L
        word_ids.append(torch.cat([item["word_ids"], torch.full((pad_amt,), pad_word_id, dtype=torch.long)]))
        char_ids.append(torch.cat([item["char_ids"], torch.full((pad_amt, max_word_len), pad_char_id, dtype=torch.long)]))
        labels.append(torch.cat([item["labels"], torch.full((pad_amt,), pad_label_id, dtype=torch.long)]))
        masks.append(torch.cat([torch.ones(L, dtype=torch.bool), torch.zeros(pad_amt, dtype=torch.bool)]))
        lengths.append(L)
    return {"word_ids": torch.stack(word_ids), "char_ids": torch.stack(char_ids),
            "labels": torch.stack(labels), "mask": torch.stack(masks),
            "lengths": torch.tensor(lengths, dtype=torch.long)}


BATCH_SIZE = 32
train_ds = NERScratchDataset(raw["train"], word_vocab, char_vocab)
val_ds   = NERScratchDataset(raw["validation"], word_vocab, char_vocab)
test_ds  = NERScratchDataset(raw["test"], word_vocab, char_vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items() if hasattr(v, "shape")})


{'word_ids': torch.Size([32, 20]), 'char_ids': torch.Size([32, 20, 20]), 'labels': torch.Size([32, 20]), 'mask': torch.Size([32, 20]), 'lengths': torch.Size([32])}


## 6. Model definitions — Char-CNN, from-scratch CRF, LSTM / BiLSTM / BiLSTM+CRF

**Why each step improves on the last:**
- **LSTM → BiLSTM**: a unidirectional LSTM only sees left context. NER
  often needs right context too, so BiLSTM encodes both directions and
  concatenates them.
- **BiLSTM → +CRF**: BiLSTM still tags each token independently (softmax),
  so it can produce illegal sequences (e.g. `I-PER` right after `O`). The
  CRF layer learns transition scores between adjacent tags and decodes the
  *globally* best sequence with the Viterbi algorithm, fixing exactly these
  boundary/consistency errors.


In [43]:
# ---- Character-level CNN (spelling features for OOV words) ---- #
class CharCNNEncoder(nn.Module):
    def __init__(self, char_vocab_size, char_emb_dim=30, num_filters=50, kernel_sizes=(3, 4, 5), dropout=0.3):
        super().__init__()
        self.char_embedding = nn.Embedding(char_vocab_size, char_emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_emb_dim, num_filters, kernel_size=k, padding=k // 2) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.output_dim = num_filters * len(kernel_sizes)

    def forward(self, char_ids):
        B, T, W = char_ids.shape
        emb = self.char_embedding(char_ids.view(B * T, W)).transpose(1, 2)
        pooled = [torch.max(torch.relu(conv(emb)), dim=2).values for conv in self.convs]
        out = self.dropout(torch.cat(pooled, dim=1))
        return out.view(B, T, self.output_dim)


# ---- From-scratch linear-chain CRF (forward algorithm + Viterbi decode) ---- #
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags) * 0.01)
        self.start_transitions = nn.Parameter(torch.randn(num_tags) * 0.01)
        self.end_transitions = nn.Parameter(torch.randn(num_tags) * 0.01)

    def forward(self, emissions, tags, mask):
        gold = self._score_sentence(emissions, tags, mask)
        Z = self._forward_alg(emissions, mask)
        return (Z - gold).mean()

    def _score_sentence(self, emissions, tags, mask):
        B, T, _ = emissions.shape
        mask = mask.float()
        score = self.start_transitions[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)
        for t in range(1, T):
            emit = emissions[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            trans = self.transitions[tags[:, t - 1], tags[:, t]]
            score = score + (trans + emit) * mask[:, t]
        seq_lens = mask.sum(1).long() - 1
        last_tags = tags.gather(1, seq_lens.unsqueeze(1)).squeeze(1)
        return score + self.end_transitions[last_tags]

    def _forward_alg(self, emissions, mask):
        B, T, C = emissions.shape
        mask = mask.float()
        alpha = self.start_transitions.unsqueeze(0) + emissions[:, 0]
        for t in range(1, T):
            broadcast = alpha.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, t].unsqueeze(1)
            new_alpha = torch.logsumexp(broadcast, dim=1)
            m = mask[:, t].unsqueeze(1)
            alpha = new_alpha * m + alpha * (1 - m)
        return torch.logsumexp(alpha + self.end_transitions.unsqueeze(0), dim=1)

    def decode(self, emissions, mask):
        B, T, C = emissions.shape
        mask = mask.bool()
        backpointers = []
        score = self.start_transitions.unsqueeze(0) + emissions[:, 0]
        for t in range(1, T):
            broadcast = score.unsqueeze(2) + self.transitions.unsqueeze(0)
            best_score, best_idx = broadcast.max(dim=1)
            new_score = best_score + emissions[:, t]
            m = mask[:, t].unsqueeze(1)
            score = new_score * m + score * (~m)
            backpointers.append(best_idx)
        score = score + self.end_transitions.unsqueeze(0)
        _, best_last_tag = score.max(dim=1)
        seq_lens = mask.sum(1)
        best_paths = []
        for b in range(B):
            L = seq_lens[b].item()
            tag = best_last_tag[b].item()
            path = [tag]
            for t in range(L - 2, -1, -1):
                tag = backpointers[t][b, tag].item()
                path.append(tag)
            path.reverse()
            best_paths.append(path)
        return best_paths


# ---- Shared input layer: pretrained word embedding + char-CNN ---- #
class _EmbeddingStack(nn.Module):
    def __init__(self, vocab_size, word_emb_dim, char_vocab_size, char_emb_dim=30,
                 char_num_filters=50, pretrained_embeddings=None, dropout=0.5):
        super().__init__()
        self.word_embedding = nn.Embedding(vocab_size, word_emb_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.word_embedding.weight.data.copy_(torch.tensor(pretrained_embeddings))
        self.char_encoder = CharCNNEncoder(char_vocab_size, char_emb_dim, char_num_filters)
        self.output_dim = word_emb_dim + self.char_encoder.output_dim
        self.dropout = nn.Dropout(dropout)

    def forward(self, word_ids, char_ids):
        x = torch.cat([self.word_embedding(word_ids), self.char_encoder(char_ids)], dim=-1)
        return self.dropout(x)


class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, num_tags, char_vocab_size, word_emb_dim=100,
                 hidden_dim=128, pretrained_embeddings=None, dropout=0.5):
        super().__init__()
        self.embed = _EmbeddingStack(vocab_size, word_emb_dim, char_vocab_size,
                                      pretrained_embeddings=pretrained_embeddings, dropout=dropout)
        self.lstm = nn.LSTM(self.embed.output_dim, hidden_dim, batch_first=True, bidirectional=False)
        self.classifier = nn.Linear(hidden_dim, num_tags)
        self.dropout = nn.Dropout(dropout)

    def forward(self, word_ids, char_ids, mask=None):
        out, _ = self.lstm(self.embed(word_ids, char_ids))
        return self.classifier(self.dropout(out))

    def loss(self, emissions, labels, mask):
        return nn.CrossEntropyLoss(ignore_index=-100)(emissions.view(-1, emissions.shape[-1]), labels.view(-1))

    def predict(self, emissions, mask):
        preds = emissions.argmax(-1)
        return [preds[b, :mask[b].sum()].tolist() for b in range(preds.shape[0])]


class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, num_tags, char_vocab_size, word_emb_dim=100,
                 hidden_dim=128, num_layers=1, pretrained_embeddings=None, dropout=0.5):
        super().__init__()
        self.embed = _EmbeddingStack(vocab_size, word_emb_dim, char_vocab_size,
                                      pretrained_embeddings=pretrained_embeddings, dropout=dropout)
        self.lstm = nn.LSTM(self.embed.output_dim, hidden_dim, num_layers=num_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if num_layers > 1 else 0)
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)
        self.dropout = nn.Dropout(dropout)

    def forward(self, word_ids, char_ids, mask=None):
        out, _ = self.lstm(self.embed(word_ids, char_ids))
        return self.classifier(self.dropout(out))

    def loss(self, emissions, labels, mask):
        return nn.CrossEntropyLoss(ignore_index=-100)(emissions.view(-1, emissions.shape[-1]), labels.view(-1))

    def predict(self, emissions, mask):
        preds = emissions.argmax(-1)
        return [preds[b, :mask[b].sum()].tolist() for b in range(preds.shape[0])]


class BiLSTMCRFTagger(nn.Module):
    def __init__(self, vocab_size, num_tags, char_vocab_size, word_emb_dim=100,
                 hidden_dim=128, num_layers=1, pretrained_embeddings=None, dropout=0.5):
        super().__init__()
        self.embed = _EmbeddingStack(vocab_size, word_emb_dim, char_vocab_size,
                                      pretrained_embeddings=pretrained_embeddings, dropout=dropout)
        self.lstm = nn.LSTM(self.embed.output_dim, hidden_dim, num_layers=num_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if num_layers > 1 else 0)
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)
        self.dropout = nn.Dropout(dropout)
        self.crf = CRF(num_tags)

    def forward(self, word_ids, char_ids, mask=None):
        out, _ = self.lstm(self.embed(word_ids, char_ids))
        return self.classifier(self.dropout(out))

    def loss(self, emissions, labels, mask):
        safe_labels = labels.clone()
        safe_labels[safe_labels == -100] = 0
        return self.crf(emissions, safe_labels, mask)

    def predict(self, emissions, mask):
        return self.crf.decode(emissions, mask)


def build_model(arch, **kwargs):
    arch = arch.lower()
    if arch == "lstm":
        return LSTMTagger(**kwargs)
    elif arch == "bilstm":
        return BiLSTMTagger(**kwargs)
    elif arch in ("bilstm_crf", "bilstm-crf"):
        return BiLSTMCRFTagger(**kwargs)
    raise ValueError(f"Unknown architecture: {arch}")

print("Models defined: LSTMTagger, BiLSTMTagger, BiLSTMCRFTagger")


Models defined: LSTMTagger, BiLSTMTagger, BiLSTMCRFTagger


## 7. Training loop + seqeval evaluation

`seqeval` scores whole entity **spans**, not individual tokens — the
correct way to grade NER (a 3-token entity with 1 wrong token counts as one
wrong prediction, not partial credit).


In [44]:
!pip install --upgrade pip setuptools wheel
!pip install seqeval --no-build-isolation

In [45]:
!pip install seqeval datasets

In [52]:
#!pip install -q seqeval
import torch
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from tqdm.auto import tqdm


def ids_to_labels(id_lists):
    return [[ID2LABEL[i] for i in seq] for seq in id_lists]

def evaluate_predictions(true_labels, pred_labels):
    # 1. حساب المقاييس الإجمالية (Overall)
    overall = {
        "precision": float(precision_score(true_labels, pred_labels)),
        "recall": float(recall_score(true_labels, pred_labels)),
        "f1": float(f1_score(true_labels, pred_labels))
    }
    
    # 2. استخراج المقاييس لكل كيان (Per Entity) من النص العادي لـ classification_report
    report_text = classification_report(true_labels, pred_labels)
    per_entity = {}
    
    lines = report_text.strip().split('\n')
    for line in lines[2:]:  # نحذف الهيدر
        line = line.strip()
        if not line or line.startswith("micro") or line.startswith("macro") or line.startswith("weighted"):
            continue
        parts = line.split()
        if len(parts) >= 5:
            etype = parts[0]
            per_entity[etype] = {
                "precision": float(parts[1]),
                "recall": float(parts[2]),
                "f1": float(parts[3]),
                "support": int(parts[4])
            }

    return {
        "overall": overall,
        "per_entity": per_entity
    }


def print_evaluation(results, model_name=""):
    print(f"\n{'=' * 60}\nEvaluation: {model_name}\n{'=' * 60}")
    o = results["overall"]
    print(f"Overall  -> P: {o['precision']:.4f}  R: {o['recall']:.4f}  F1: {o['f1']:.4f}")
    print(f"{'Entity':<8}{'Precision':>12}{'Recall':>12}{'F1':>12}{'Support':>12}")
    for etype, m in results["per_entity"].items():
        print(f"{etype:<8}{m['precision']:>12.4f}{m['recall']:>12.4f}{m['f1']:>12.4f}{m['support']:>12}")


def run_epoch(model, loader, optimizer=None, device="cpu"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_gold = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for batch in tqdm(loader, leave=False):
            word_ids, char_ids = batch["word_ids"].to(device), batch["char_ids"].to(device)
            labels, mask = batch["labels"].to(device), batch["mask"].to(device)
            emissions = model(word_ids, char_ids, mask)
            loss = model.loss(emissions, labels, mask)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            total_loss += loss.item()
            preds = model.predict(emissions, mask)
            for b in range(labels.shape[0]):
                L = mask[b].sum().item()
                all_gold.append(labels[b, :L].tolist())
                all_preds.append(preds[b][:L])
    return total_loss / len(loader), all_gold, all_preds


def train_scratch_model(arch, epochs=10, hidden_dim=128, lr=1e-3):
    model = build_model(arch, vocab_size=len(word_vocab), num_tags=len(LABEL_LIST),
                         char_vocab_size=len(char_vocab), word_emb_dim=WORD_EMB_DIM,
                         hidden_dim=hidden_dim, pretrained_embeddings=embedding_matrix).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state = -1, None
    for epoch in range(1, epochs + 1):
        train_loss, _, _ = run_epoch(model, train_loader, optimizer, device)
        val_loss, val_gold, val_preds = run_epoch(model, val_loader, None, device)
        metrics = evaluate_predictions(ids_to_labels(val_gold), ids_to_labels(val_preds))
        f1 = metrics["overall"]["f1"]
        print(f"[{arch}] epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_F1={f1:.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    _, test_gold, test_preds = run_epoch(model, test_loader, None, device)
    test_metrics = evaluate_predictions(ids_to_labels(test_gold), ids_to_labels(test_preds))
    print_evaluation(test_metrics, model_name=arch.upper())
    return model, test_metrics

print("Training utilities ready.")

Training utilities ready.


## 8. Train LSTM, BiLSTM, and BiLSTM+CRF

In [53]:
EPOCHS = 15  # increase to 15-20 for real CoNLL-2003 training
lstm_model, lstm_metrics = train_scratch_model("lstm", epochs=EPOCHS)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[lstm] epoch 01 | train_loss=1.6290 val_loss=1.2517 val_F1=0.0000


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[lstm] epoch 15 | train_loss=0.0905 val_loss=0.0393 val_F1=1.0000


  0%|          | 0/2 [00:00<?, ?it/s]


Evaluation: LSTM
Overall  -> P: 1.0000  R: 1.0000  F1: 1.0000
Entity     Precision      Recall          F1     Support
MISC          1.0000      1.0000      1.0000          32
ORG           1.0000      1.0000      1.0000          32
LOC           1.0000      1.0000      1.0000          34
PER           1.0000      1.0000      1.0000          28


In [54]:
bilstm_model, bilstm_metrics = train_scratch_model("bilstm", epochs=EPOCHS)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[bilstm] epoch 01 | train_loss=1.4935 val_loss=1.3448 val_F1=0.0000


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[bilstm] epoch 15 | train_loss=0.0147 val_loss=0.0028 val_F1=1.0000


  0%|          | 0/2 [00:00<?, ?it/s]


Evaluation: BILSTM
Overall  -> P: 1.0000  R: 1.0000  F1: 1.0000
Entity     Precision      Recall          F1     Support
MISC          1.0000      1.0000      1.0000          32
ORG           1.0000      1.0000      1.0000          32
LOC           1.0000      1.0000      1.0000          34
PER           1.0000      1.0000      1.0000          28


In [55]:
bilstm_crf_model, bilstm_crf_metrics = train_scratch_model("bilstm_crf", epochs=EPOCHS)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[bilstm_crf] epoch 01 | train_loss=18.6680 val_loss=16.9632 val_F1=0.0000


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[bilstm_crf] epoch 15 | train_loss=0.1179 val_loss=0.0178 val_F1=1.0000


  0%|          | 0/2 [00:00<?, ?it/s]


Evaluation: BILSTM_CRF
Overall  -> P: 1.0000  R: 1.0000  F1: 1.0000
Entity     Precision      Recall          F1     Support
MISC          1.0000      1.0000      1.0000          32
ORG           1.0000      1.0000      1.0000          32
LOC           1.0000      1.0000      1.0000          34
PER           1.0000      1.0000      1.0000          28


## 9. Fine-tune a Transformer (HuggingFace)

Uses `AutoTokenizer` + `AutoModelForTokenClassification` + `Trainer`.
Requires internet (Hub) and ideally a GPU.


In [62]:
!pip install -q seqeval datasets transformers

TRANSFORMER_MODEL_NAME = "distilbert-base-cased"   # or "bert-base-cased"
TRANSFORMER_EPOCHS = 6

try:
    import numpy as np
    from datasets import Dataset as HFDataset, DatasetDict
    from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                               DataCollatorForTokenClassification, TrainingArguments, Trainer)
    # استخدام seqeval العادية المباشرة من غير v1 خالص
    from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

    def tokenize_and_align_labels(examples, tokenizer):
        tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, label in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)
            prev = None
            label_ids = []
            for wid in word_ids:
                if wid is None:
                    label_ids.append(-100)
                elif wid != prev:
                    label_ids.append(label[wid])
                else:
                    label_ids.append(-100)
                prev = wid
            all_labels.append(label_ids)
        tokenized["labels"] = all_labels
        return tokenized

    def build_compute_metrics():
        def compute_metrics(eval_pred):
            predictions, labels = eval_pred
            predictions = np.argmax(predictions, axis=2)
            true_labels, true_preds = [], []
            for pred_seq, label_seq in zip(predictions, labels):
                cur_t, cur_p = [], []
                for p, l in zip(pred_seq, label_seq):
                    if l == -100:
                        continue
                    cur_t.append(ID2LABEL[l]); cur_p.append(ID2LABEL[p])
                true_labels.append(cur_t); true_preds.append(cur_p)
            
            # حساب المقاييس العامة مباشرة عبر seqeval العادية
            result = {
                "precision": float(precision_score(true_labels, true_preds)),
                "recall": float(recall_score(true_labels, true_preds)),
                "f1": float(f1_score(true_labels, true_preds))
            }

            # قراءة التقرير النصي واستخراج الـ F1 لكل entity من seqeval العادية
            report_text = classification_report(true_labels, true_preds)
            for line in report_text.strip().split('\n')[2:]:
                parts = line.strip().split()
                if len(parts) >= 4 and parts[0] in ENTITY_TYPES:
                    result[f"{parts[0]}_f1"] = float(parts[3])

            return result
        return compute_metrics

    tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)
    transformer_model = AutoModelForTokenClassification.from_pretrained(
        TRANSFORMER_MODEL_NAME, num_labels=len(LABEL_LIST),
        id2label=ID2LABEL, label2id={v: k for k, v in ID2LABEL.items()},
    )

    hf_ds = DatasetDict({
        split: HFDataset.from_list(raw[split]) for split in ["train", "validation", "test"]
    })

    tokenized_ds = DatasetDict({
        split: hf_ds[split].map(lambda ex: tokenize_and_align_labels(ex, tokenizer),
                                 batched=True, remove_columns=hf_ds[split].column_names)
        for split in ["train", "validation", "test"]
    })

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    
    training_args = TrainingArguments(
        output_dir="checkpoints/transformer", eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=TRANSFORMER_EPOCHS, weight_decay=0.01,
        load_best_model_at_end=True, metric_for_best_model="f1", logging_steps=50, report_to=[],
    )
    
    trainer = Trainer(
        model=transformer_model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=build_compute_metrics()
    )
                       
    trainer.train()

    raw_test_metrics = trainer.evaluate(tokenized_ds["test"])
    transformer_metrics = {
        "overall": {"precision": raw_test_metrics.get("eval_precision", 0.0),
                    "recall": raw_test_metrics.get("eval_recall", 0.0),
                    "f1": raw_test_metrics.get("eval_f1", 0.0)},
        "per_entity": {etype: {"f1": raw_test_metrics.get(f"eval_{etype}_f1", 0.0),
                                "precision": None, "recall": None, "support": None}
                        for etype in ENTITY_TYPES},
    }
    print("Transformer test metrics:", transformer_metrics["overall"])
except Exception as e:
    print(f"Transformer fine-tuning skipped/failed ({type(e).__name__}: {str(e)[:200]}).")
    print("This step needs the 'datasets'/'transformers' packages plus internet access to the "
          "HuggingFace Hub. Run `pip install datasets transformers seqeval` and enable internet "
          "(Colab/Kaggle: Settings -> Internet -> On) to fine-tune a real transformer.")
    transformer_model, tokenizer, transformer_metrics = None, None, None

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Transformer test metrics: {'precision': 0.8345864661654135, 'recall': 0.8809523809523809, 'f1': 0.8571428571428571}


## 10. Compare all four architectures

In [63]:
rows = []
for name, res in [("LSTM", lstm_metrics), ("BiLSTM", bilstm_metrics),
                   ("BiLSTM + CRF", bilstm_crf_metrics), ("Transformer", transformer_metrics)]:
    if res is None:
        continue
    row = {"Model": name, "Precision": res["overall"]["precision"],
           "Recall": res["overall"]["recall"], "F1": res["overall"]["f1"]}
    for etype, m in res["per_entity"].items():
        row[f"{etype}_F1"] = m["f1"]
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index("Model").round(4)
display(comparison_df)

print('''
Where BiLSTM + CRF improves over plain BiLSTM
-----------------------------------------------
A plain BiLSTM makes an INDEPENDENT softmax decision at every token, so it
cannot enforce legal/consistent tag sequences. This causes two common
errors: (1) illegal transitions like O -> I-ORG with no B-ORG start, and
(2) split/merged multi-word entities (e.g. tagging "New" as B-LOC but
"York" as O). The CRF layer adds a learned transition matrix and decodes
the WHOLE sequence jointly via Viterbi, so illegal paths get a strongly
negative score during training. This typically shows up as higher
per-entity precision AND recall, especially for multi-token types like
ORG and MISC.
''')


,Precision,Recall,F1,MISC_F1,ORG_F1,LOC_F1,PER_F1
Model,,,,,,,
LSTM,1.0000,1.000,1.0000,1.00,1.00,1.00,1.00
BiLSTM,1.0000,1.000,1.0000,1.00,1.00,1.00,1.00
BiLSTM + CRF,1.0000,1.000,1.0000,1.00,1.00,1.00,1.00
Transformer,0.8346,0.881,0.8571,0.91,0.93,0.79,0.81



Where BiLSTM + CRF improves over plain BiLSTM
-----------------------------------------------
A plain BiLSTM makes an INDEPENDENT softmax decision at every token, so it
cannot enforce legal/consistent tag sequences. This causes two common
errors: (1) illegal transitions like O -> I-ORG with no B-ORG start, and
(2) split/merged multi-word entities (e.g. tagging "New" as B-LOC but
"York" as O). The CRF layer adds a learned transition matrix and decodes
the WHOLE sequence jointly via Viterbi, so illegal paths get a strongly
negative score during training. This typically shows up as higher
per-entity precision AND recall, especially for multi-token types like
ORG and MISC.



## 11. Visualize predicted entity spans (color-coded, inline)

Uses the trained BiLSTM+CRF model already in memory.


In [64]:
ENTITY_COLORS = {"PER": "#FF6B6B", "ORG": "#4ECDC4", "LOC": "#FFD93D", "MISC": "#A78BFA"}
TOKEN_RE = re.compile(r"\w+|[^\w\s]")

def simple_tokenize(text):
    return TOKEN_RE.findall(text)

def predict_scratch(text, model):
    tokens = simple_tokenize(text)
    if not tokens:
        return []
    word_ids = torch.tensor([word_vocab.encode(tokens)], dtype=torch.long).to(device)
    char_ids = torch.tensor([char_vocab.encode_sentence(tokens)], dtype=torch.long).to(device)
    mask = torch.ones(1, len(tokens), dtype=torch.bool).to(device)
    model.eval()
    with torch.no_grad():
        emissions = model(word_ids, char_ids, mask)
        pred_ids = model.predict(emissions, mask)[0]
    return list(zip(tokens, [ID2LABEL[i] for i in pred_ids]))

def iob_to_spans(tagged):
    chunks, cur_words, cur_type = [], [], None
    for word, tag in tagged:
        if tag == "O":
            if cur_words:
                chunks.append((" ".join(cur_words), cur_type)); cur_words, cur_type = [], None
            chunks.append((word, None))
        else:
            etype = tag.split("-")[-1]
            is_begin = tag.startswith("B-") or etype != cur_type
            if is_begin and cur_words:
                chunks.append((" ".join(cur_words), cur_type)); cur_words = []
            cur_words.append(word); cur_type = etype
    if cur_words:
        chunks.append((" ".join(cur_words), cur_type))
    return chunks

def render_html(text, model):
    chunks = iob_to_spans(predict_scratch(text, model))
    html = ""
    for word, etype in chunks:
        if etype:
            color = ENTITY_COLORS.get(etype, "#ccc")
            html += (f'<span style="background:{color};padding:2px 6px;border-radius:6px;'
                      f'margin:0 2px;font-weight:600;">{word} '
                      f'<sup style="font-size:10px;">{etype}</sup></span> ')
        else:
            html += word + " "
    return html

sample_texts = [
    "Barack Obama met Angela Merkel in Berlin during the G7 Summit.",
    "Google announced a partnership with the United Nations.",
]
for t in sample_texts:
    display(HTML(render_html(t, bilstm_crf_model)))


## 12. Launch the Gradio deployment app

A self-contained, professional Gradio UI that uses the models already
trained in this notebook session — no external files needed. Lets you pick
an architecture, type text, and see entities highlighted live.


In [66]:
# !pip install -q gradio
import gradio as gr

def predict_transformer(text):
    tokens = simple_tokenize(text)
    if not tokens:
        return []
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        logits = transformer_model(**enc).logits
    pred_ids = logits.argmax(-1)[0].tolist()
    word_ids = enc.word_ids(batch_index=0)
    id2label = transformer_model.config.id2label
    results, seen = [], set()
    for wid, pid in zip(word_ids, pred_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        results.append((tokens[wid], id2label[pid]))
    return results


MODEL_REGISTRY = {"LSTM (scratch)": ("scratch", lstm_model),
                   "BiLSTM (scratch)": ("scratch", bilstm_model),
                   "BiLSTM + CRF (scratch)": ("scratch", bilstm_crf_model)}
if transformer_model is not None:
    MODEL_REGISTRY["Fine-tuned Transformer"] = ("transformer", transformer_model)


def run_ner(text, model_label):
    if not text or not text.strip():
        return {"text": "", "entities": []}, pd.DataFrame(columns=["Type", "Entities Found", "Count"]), \
               "Type or paste some text above, then click **Analyze**."
    kind, model = MODEL_REGISTRY[model_label]
    tagged = predict_scratch(text, model) if kind == "scratch" else predict_transformer(text)
    chunks = iob_to_spans(tagged)
    highlighted = [(word, etype) for word, etype in chunks]

    counts, entity_words = Counter(), {}
    for word, etype in chunks:
        if etype:
            counts[etype] += 1
            entity_words.setdefault(etype, []).append(word)

    ENTITY_NAMES = {"PER": "Person", "ORG": "Organization", "LOC": "Location", "MISC": "Miscellaneous"}
    rows = [{"Type": f"{ENTITY_NAMES[e]} ({e})", "Entities Found": ", ".join(dict.fromkeys(entity_words[e])),
             "Count": counts[e]} for e in ENTITY_TYPES if counts[e] > 0]
    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Type", "Entities Found", "Count"])
    status = f"Found **{sum(counts.values())}** entities across **{len(counts)}** categories using **{model_label}**."
    return highlighted, df, status


CUSTOM_CSS = '''
.gradio-container { font-family: 'Inter','Segoe UI',system-ui,sans-serif !important; max-width: 1180px !important; margin: 0 auto !important; }
#hero { background: linear-gradient(135deg,#1e1b4b 0%,#4c1d95 45%,#6d28d9 100%); border-radius: 20px; padding: 36px 40px; color: white; margin-bottom: 22px; box-shadow: 0 10px 40px rgba(76,29,149,0.25); }
#hero h1 { font-size: 2.1rem; font-weight: 800; margin: 0 0 6px 0; letter-spacing: -0.02em; }
#hero p { font-size: 1.02rem; opacity: 0.9; margin: 0; }
.legend-chip { display:inline-flex; align-items:center; gap:6px; padding:5px 14px; border-radius:999px; font-size:0.85rem; font-weight:600; margin-right:8px; color:#1a1a1a; }
#analyze-btn { background: linear-gradient(135deg,#6d28d9,#4c1d95) !important; color:white !important; font-weight:700 !important; border-radius:12px !important; border:none !important; }
footer { display:none !important; }
'''

EXAMPLES = [
    "Barack Obama met Angela Merkel in Berlin during the G7 Summit last October.",
    "Google announced a new partnership with the United Nations to fight climate change.",
    "Marie Curie won the Nobel Prize for her research conducted in Paris, France.",
]

# التعديل الرئيسي: نقل css و theme إلى gr.Blocks
with gr.Blocks(
    title="Named Entity Recognition Studio", 
    css=CUSTOM_CSS, 
    theme=gr.themes.Soft(primary_hue="violet", secondary_hue="teal")
) as demo:
    gr.HTML('''
    <div id="hero">
        <h1>🔍 Named Entity Recognition Studio</h1>
        <p>Compare LSTM, BiLSTM, BiLSTM+CRF, and fine-tuned Transformer architectures — entities highlighted live.</p>
        <div style="margin-top:16px;">
            <span class="legend-chip" style="background:#FF6B6B;">● PERSON</span>
            <span class="legend-chip" style="background:#4ECDC4;">● ORGANIZATION</span>
            <span class="legend-chip" style="background:#FFD93D;">● LOCATION</span>
            <span class="legend-chip" style="background:#A78BFA;">● MISC</span>
        </div>
    </div>
    ''')
    with gr.Row():
        with gr.Column(scale=1):
            model_dd = gr.Dropdown(choices=list(MODEL_REGISTRY.keys()), value=list(MODEL_REGISTRY.keys())[-1],
                                    label="🧠 Model architecture")
            text_in = gr.Textbox(label="✍️ Input text", placeholder="Paste a sentence...", lines=6)
            with gr.Row():
                analyze_btn = gr.Button("⚡ Analyze", elem_id="analyze-btn", variant="primary", scale=2)
                gr.ClearButton([text_in], value="Clear", scale=1)
            gr.Examples(examples=EXAMPLES, inputs=text_in, label="Try an example")
    status_md = gr.Markdown()
    gr.Markdown("### 🎯 Detected Entities")
    highlighted_out = gr.HighlightedText(color_map={"PER": "#FF6B6B", "ORG": "#4ECDC4", "LOC": "#FFD93D", "MISC": "#A78BFA"})
    gr.Markdown("### 📊 Entity Summary")
    table_out = gr.Dataframe(headers=["Type", "Entities Found", "Count"], wrap=True)

    analyze_btn.click(run_ner, [text_in, model_dd], [highlighted_out, table_out, status_md])
    text_in.submit(run_ner, [text_in, model_dd], [highlighted_out, table_out, status_md])

# الإطلاق بدون تمرير css داخل launch
demo.launch(share=True)

/tmp/ipykernel_58/1163842820.py:70: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_58/1163842820.py:70: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://adcbd63a355108d4f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
